# 05. [Retrieval 1] OpenAI 임베딩 모델·Chroma 설정 확정
  임베딩 모델 설정 확정 부분

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [6]:
import time # 속도 측정
from langchain_openai import OpenAIEmbeddings # 임베딩 모델

# 임베딩 모델 생성
models = {
    "OpenAI-Small": OpenAIEmbeddings(model="text-embedding-3-small"),
    "OpenAI-Large": OpenAIEmbeddings(model="text-embedding-3-large")
}

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.metrics.pairwise import cosine_similarity

# 1. 가상 데이터 로드
chunks = []
with open('../samples/processed/sample_chunks.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line))

texts = [chunk["text"] for chunk in chunks]
query = "공공 AI 학습지원 플랫폼의 총사업예산과 수행 기간은?" # 테스트 질문

# 테스트 로직
results = []
for name, model in models.items():
    # 임베딩 생성 및 시간 측정
    start_time = time.time()
    chunk_embeddings = model.embed_documents(texts)
    query_embedding = model.embed_query(query)
    duration = time.time() - start_time
    
    # 유사도 계산 (가장 높은 점수 확인)
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    best_idx = np.argmax(similarities)
    
    results.append({
        "Model": name,
        "Time(sec)": round(duration, 4), # 소요시간, 0.0001초단위로 반올림
        "Best_Score": round(similarities[best_idx], 4), # 유사도 점수, 0.0001단위로 반올림
        "Best_Chunk": f"Chunk_{best_idx+1}" # 가장 유사한 Chunk ID
    })


In [ ]:
df_results = pd.DataFrame(results)
print(df_results)

In [ ]:
# Small 모델 토큰 / 비용
from tiktoken import encoding_for_model

enc = encoding_for_model("text-embedding-3-small")
tokens = sum(len(enc.encode(t)) for t in texts)
cost = tokens / 1000 * 0.00002  # small 모델 가격 (구글링))
print(f" Small 모델 토큰 수: {tokens}, 비용: {cost}")

In [ ]:
# large 모델 토큰 / 비용
enc = encoding_for_model("text-embedding-3-large")
tokens = sum(len(enc.encode(t)) for t in texts)
cost = tokens / 1000 * 0.00013  # large 모델 가격 (구글링)
print(f" Large 모델 토큰 수: {tokens}, 비용: {cost}")

In [ ]:
query = "공공 AI 학습지원 플랫폼의 총사업예산과 수행 기간은?"

# 1. 모델 설정 (Best Score가 높은 Small 모델)
model = OpenAIEmbeddings(model="text-embedding-3-small")
query_embedding = model.embed_query(query)
chunk_embeddings = model.embed_documents(texts)

# 2. 유사도 계산
similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]

# 3. 가장 유사도가 높은 청크의 인덱스 찾기
best_idx = np.argmax(similarities)

# 4. 상세 내용 출력
print(f"- 청크 ID: {chunks[best_idx]['chunk_id']}")
print(f"- 유사도 점수: {round(similarities[best_idx], 4)}")
print(f"- 본문 내용:\n{chunks[best_idx]['text']}")

# 06. [Retrieval 1] OpenAI + Chroma 기본 유사도 검색 구현


In [ ]:
import os
import sys
import yaml
# 노트북 폴더(notebook/)에서 상위 루트 경로를 인식시키기 위함
sys.path.append("../")
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 1. 실제 내장된 YAML 파일 읽기
yaml_path = "../config/default.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 잘 로드됐는지 정보 확인 
print(f"-> active_profile: {config['retrieval']['active_profile']}")
print(f"-> 기본 검색 개수(top_k): {config['retrieval']['top_k']}") # yaml에서 수정할때마다 바뀜
print(f"-> 샘플 청크 경로: {config['paths']['chunks']}")

In [ ]:
# 3. 환경변수에 API Key가 잘 들어있는지 최종 확인
if "OPENAI_API_KEY" not in os.environ:
    print(" 경고: OPENAI_API_KEY가 설정되어 있지 않습니다. 검색 시 에러가 발생할 수 있습니다.")

# 4. 리트리버 객체 생성 (전체 config를 그대로 전달)
retriever = OpenAIChromaRetriever(config=config)

# 5. 실제 문서 내용에 있을 법한 키워드로 테스트 질문 투척!
test_query = "사업 제안서 서류 제출 마감 기한과 필수 제출 서류 목록이 어떻게 되나요?" 
print(f"테스트 질문: '{test_query}'\n")

# 6. 검색 실행
results = retriever.search(test_query)

# 7. 출력 결과 확인
print(f"검색된 근거 청크 개수: {len(results)}개 \n") #YAML에 설정된 top_k 만큼 나옵니다
for idx, res in enumerate(results, 1):
    print(f"[{idx}순위 근거]")
    print(f"- chunk_id: {res.chunk_id}")
    print(f"- doc_id: {res.doc_id}")
    print(f"- score (거리 값): {res.score:.4f}")
    print(f"- 본문 텍스트: {res.text}")
    print("-" * 60)

# 07. [Retrieval 1] OpenAI 메타데이터 필터링 구현

In [ ]:
import yaml
import os
import sys
sys.path.append("../")
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 1. 설정 로드
yaml_path = "../config/default.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. 리트리버 초기화
retriever = OpenAIChromaRetriever(config=config)
test_query = "사업명이 뭐야?"

In [ ]:
print("== 필터 없이 기본 검색 ==")

test_query = "사업명이 뭐야?"
results = retriever.search(query=test_query)
for r in results:
    print(f"- [score: {r.score:.2f}] agency: {r.metadata.get('agency')}")
    print(f"  [본문 미리보기]: {r.text[:150]}...\n")

== 필터 없이 기본 검색 ==


- [score: 1.47] agency: 가상디지털진흥원
  [본문 미리보기]: [가상 제안요청서] 이 문서는 RAG 시스템 개발과 검색 성능 실험을 위해 작성한 가상 데이터입니다. 실제 기관, 사업 또는 입찰 공고와 관련이 없습니다. 1. 사업 개요 사업명은 "2026년 공공 AI 학습지원 플랫폼 구축 사업"이며 발주기관은 가상디지털진흥원이다. ...

- [score: 1.50] agency: 가상디지털진흥원
  [본문 미리보기]: REST API 방식으로 연계하고, 매일 오전 2시에 변경된 사용자 정보를 동기화해야 한다. 데이터 이전 결과는 전체 건수, 성공 건수, 실패 건수로 구분한 검증 보고서로 제출해야 한다. 4. 성능 및 품질 요구사항 동시 접속자 1,000명 환경에서 일반 화면의 평균 ...

- [score: 1.50] agency: 가상디지털진흥원
  [본문 미리보기]: 이해도, 수행 방안, 인력 구성, 데이터 이전 계획, 보안 대책이 포함된다. 7. 산출물 착수계획서, 요구사항정의서, 화면설계서, API 명세서, 데이터 이전 결과서, 시험 결과서, 관리자 및 사용자 매뉴얼, 보안 점검 결과서, 완료보고서를 제출해야 한다. 모든 산출물...



현재 샘플 데이터의 청크 3개 모두 agency가 가상디지털진흥원이기에 결과는 이렇게 나옴 

In [ ]:
print("== agency 검색 ==")

results_agency = retriever.search(query=test_query, filters={"agency": "     가상디지털진흥원 "})

for res in results_agency:
    print(f"- [score: {res.score:.2f}] agency: {res.metadata.get('agency')}")

== agency 검색 ==


- [score: 1.47] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원


strip()을 추가하여 앞뒤 띄어쓰기가 있어도 검색 됨

In [ ]:
print("== agency 부분 검색 ==")

results_agency = retriever.search(query=test_query, filters={"agency": " 가상 "})

for res in results_agency:
    print(f"- [score: {res.score:.2f}] agency: {res.metadata.get('agency')}")

== agency 부분 검색 ==
- [score: 1.47] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원
- [score: 1.50] agency: 가상디지털진흥원


"가상" 으로만 검색해도 가상디지털진흥원의 부분 일치된 청크가 검색되는 모습

In [ ]:
print("== doc_id 정확도 필터 ('sample_rfp')==")

results_doc = retriever.search(query=test_query, filters={"doc_id": "sample_rfp"})

for res in results_doc:
    print(f"- [score: {res.score:.2f}] doc_id: {res.doc_id}")

== doc_id 정확도 필터 ('sample_rfp')==
- [score: 1.47] doc_id: sample_rfp
- [score: 1.50] doc_id: sample_rfp
- [score: 1.50] doc_id: sample_rfp


현재 청크된 샘플 데이터 모두 sample_rfp가 doc_id 기에 결과는 이렇게

In [ ]:
print("== 필터 미일치 (존재하지 않는 기관) ==")

results_empty = retriever.search(query=test_query, filters={"agency": "우주항공청"})

print(f"결과 개수: {len(results_empty)}개")

== 필터 미일치 (존재하지 않는 기관) ==
안내: '{'agency': '우주항공청'}' 조건에 일치하는 문서를 찾을 수 없습니다.
결과 개수: 0개


우주 항공청이라는 agency는 샘플에 없으니 결과는 0개

In [ ]:
print("== 사업명 검색 ==")

results_title = retriever.search(query=test_query, filters={"title": "2026년 공공 AI 학습지원 플랫폼"})

for res in results_title:
    project_title = res.metadata.get("title", "N/A")
    print(f"- [score: {res.score:.2f}] doc_id: {res.doc_id} | title: {project_title}")

== 사업명 검색 ==


- [score: 1.47] doc_id: sample_rfp | title: 2026년 공공 AI 학습지원 플랫폼 구축 사업
- [score: 1.50] doc_id: sample_rfp | title: 2026년 공공 AI 학습지원 플랫폼 구축 사업
- [score: 1.50] doc_id: sample_rfp | title: 2026년 공공 AI 학습지원 플랫폼 구축 사업


# 16. [Retrieval 1] OpenAI 임베딩·Chroma 인덱스 구축

In [1]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [3]:
# 1. DB 적재 스크립트 실행
import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 10222개의 청크 확인
적재 전 Chroma DB 청크 수: 0개
[1/11] 0 ~ 1000번째 청크 저장 중
[2/11] 1000 ~ 2000번째 청크 저장 중
[3/11] 2000 ~ 3000번째 청크 저장 중
[4/11] 3000 ~ 4000번째 청크 저장 중
[5/11] 4000 ~ 5000번째 청크 저장 중
[6/11] 5000 ~ 6000번째 청크 저장 중
[7/11] 6000 ~ 7000번째 청크 저장 중
[8/11] 7000 ~ 8000번째 청크 저장 중
[9/11] 8000 ~ 9000번째 청크 저장 중
[10/11] 9000 ~ 10000번째 청크 저장 중
[11/11] 10000 ~ 10222번째 청크 저장 중
최종 Chroma DB에 저장 된 청크 수: 10222개
모든 청크가 성공적으로 vector_db/openai에 저장 되었습니다.
적재 스크립트 실행 완료!


In [6]:
import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 1. 설정 파일 로드
yaml_path = "../config/default.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. Retriever 초기화
retriever = OpenAIChromaRetriever(config=config)

# 3. 실험용 쿼리 및 필터 설정
test_query = "재난 안전 관리 시스템 및 데이터 분석 구축 용역"
test_filters = {"notice_round": 0}  # 👈 실제 메타데이터 한글 Key와 일부 단어 지정!

# 4. 검색 실행
results = retriever.search(query=test_query, filters=test_filters)

# 5. 결과 확인
print(f"총 {len(results)}건의 문서를 찾았습니다.\n")
for i, res in enumerate(results):
    print(f"--- [결과 {i+1}] Score: {res.score:.4f} ---")
    print(f"메타데이터: {res.metadata}")
    print(f"본문 미리보기: {res.text[:150]}...\n")

총 5건의 문서를 찾았습니다.

--- [결과 1] Score: 0.7485 ---
메타데이터: {'summary': '- 종량제봉투 판매관리 전산시스템 개선: 기존 시스템의 노후화에 따른 문제점을 분석하여 효율적인 판매관리 및 재고/통계자료 산출을 개선함\n- 인터넷 주문, 결제 시스템 구축: 가상계좌 결제 시스템 도입, 인터넷 주문시스템 구축으로 맞춤형 서비스 제공 및 지정판매소 위치조회를 통한 시민 편의성 증대\n- 종량제물품 지정판매소 위치조회서비스 구축: 최근 3개월간 구매이력을 통해 지정판매소 위치 조회 서비스 제공\n- 경영분석 Chart 서비스 구축: 누적된 판매 데이터를 분석하여 다양한 그래프(차트) 조회 서비스 제공\n- 소프트웨어 사업자로 등록한 업체로서 자격조건 충족하여 참여 가능', 'title': '종량제봉투 판매관리 전산시스템 개선사업', 'document_type': 'hwp', 'source_exists': True, 'file_type': 'hwp', 'budget': '55000000.0', 'project_name': '종량제봉투 판매관리 전산시스템 개선사업', 'published_at': '2024-05-31 13:54:33', 'file_name': '파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp', 'doc_id': 'doc_066', 'page': '', 'agency': '파주도시관광공사', 'bid_start_at': '2024-05-31 14:00:00', 'chunk_id': 'doc_066_chunk_0084', 'source_path': '/data/original_data/files/파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업.hwp', 'notice_round': '0.0', 'bid_end_at': '2024-06-11 14:00:00', 'notice_number': '20240541779'}
본문 미리보기: 료관리 대장’을 작성, 인계자ㆍ인수자가 직접 서명한 후 제공하고 

In [4]:
import sys
sys.path.append("../")
import yaml
from pathlib import Path
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 설정 파일 로드
yaml_path = "../config/default.yaml"
with open(yaml_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Retriever 생성 (chunks 불필요, config만 전달)
retrieval_config = config.get("retrieval", {})
retriever = OpenAIChromaRetriever(retrieval_config)

# 검색 쿼리 실행
query = "시스템 구축 사업"
results = retriever.search(query=query, top_k=1)

# 검색 결과가 있는지 확인
if not results:
    print("검색 결과가 없습니다. DB가 비어있는지 확인하세요.")
else:
    top_result = results[0]
    
    print("=== Retriever 반환 규격(SearchResult) 검증 ===")
    print(f"1. chunk_id [type: {type(top_result.chunk_id).__name__}] : {top_result.chunk_id}")
    print(f"2. doc_id   [type: {type(top_result.doc_id).__name__}]   : {top_result.doc_id}")
    print(f"3. score    [type: {type(top_result.score).__name__}]  : {top_result.score}")
    
    print(f"4. metadata [type: {type(top_result.metadata).__name__}] :")
    for key, value in top_result.metadata.items():
        print(f"   - {key}: {value}")
        
    print(f"\n5. text     [type: {type(top_result.text).__name__}]   :")
    print(f"   {top_result.text[:100]}... (이하 생략)")

=== Retriever 반환 규격(SearchResult) 검증 ===
1. chunk_id [type: str] : doc_094_chunk_0010
2. doc_id   [type: str]   : doc_094
3. score    [type: float]  : 0.7110670208930969
4. metadata [type: dict] :
   - notice_number: 20240408682
   - agency: 그랜드코리아레저(주)
   - file_type: hwp
   - document_type: hwp
   - page: 
   - bid_start_at: 2024-04-04 18:00:00
   - published_at: 2024-04-04 14:17:16
   - bid_end_at: 2024-05-20 11:00:00
   - summary: - 사업개요: GKL 그룹웨어 시스템 구축사업
- 추진배경: 그룹웨어 및 기록물관리, 사내SNS 시스템 노후화로 인한 개선 필요, 정부권장정책 적용, 웹기반 시스템 필요
- 사업범위: 그룹웨어 시스템 구축, 기록물관리 시스템 구축, 업무용 메신저 시스템 구축
- 기대효과: 시스템 안정성 확보, 관리 효율성 향상, 사용자 편의 증대
   - file_name: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp
   - notice_round: 0.0
   - doc_id: doc_094
   - source_path: /data/original_data/files/그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp
   - title: 2024년도 GKL  그룹웨어 시스템 구축 용역
   - project_name: 2024년도 GKL  그룹웨어 시스템 구축 용역
   - source_exists: True
   - budget: 1515000000.0
   - chunk_id: doc_094_chunk_0010

5. text     [t

In [ ]:
# relavance score 점수가 출력 되도록 변경 

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 1. DB 연결 (VM 경로)
vectorstore = Chroma(
    collection_name="bidmate_openai",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="/data/processed/vector_db/openai"
)

# 2. 검색 테스트 (같은 질문 2번 실행)
query = "차세대 학사 정보시스템 구축에 대해 알려줘"
res = vectorstore.similarity_search_with_relevance_scores(query, k=3)

print("=== 검색 결과 ===")
for doc, score in res:
    print(f"ID: {doc.metadata['chunk_id']} | Score: {score:.4f}")
    print(f"Content: {doc.page_content[:100]}...\n")

=== 검색 결과 ===
ID: doc_008_chunk_0045 | Score: 0.3854
Content: 의 차세대 정보시스템 구축을 위한 표준 프로세스 정립 세부 내용 ❍ 표준 업무프로세스 정립 및 차세대 개발 적용 - 차세대 학사행정시스템의 정보화 수준 향상을 위해 시스템 구축과 ...

ID: doc_008_chunk_0016 | Score: 0.3299
Content: 통합 학사행정시스템 구축 ❍ 서울, 세종 캠퍼스 학사제도를 아우르는 표준화된 학사행정시스템 구축 필요 ❍ 핵심역량과 연계된 강의계획서 등록 등 일부 학사 기능이 양 캠퍼스(서울, ...

ID: doc_080_chunk_0001 | Score: 0.3227
Content: 차세대 ERP시스템 구축 사업 제 안 요 청 서 2024. 6. 목 차 Ⅰ. 사업 개요 1. 사업명 謸 2. 사업기간 蕜 3. 사업예산 蕜 4. 계약방법 蕜 Ⅱ. 주요 시스템 구축...



VM 경로에서 가져온 정보로 확인

In [2]:
import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 10222개의 청크 확인
적재 전 Chroma DB 청크 수: 0개
추가(변경)된 청크 10222개 저장 중
[1/11] 0 ~ 1000번째 대상 처리 중
[2/11] 1000 ~ 2000번째 대상 처리 중
[3/11] 2000 ~ 3000번째 대상 처리 중
[4/11] 3000 ~ 4000번째 대상 처리 중
[5/11] 4000 ~ 5000번째 대상 처리 중
[6/11] 5000 ~ 6000번째 대상 처리 중
[7/11] 6000 ~ 7000번째 대상 처리 중
[8/11] 7000 ~ 8000번째 대상 처리 중
[9/11] 8000 ~ 9000번째 대상 처리 중
[10/11] 9000 ~ 10000번째 대상 처리 중
[11/11] 10000 ~ 10222번째 대상 처리 중
== VectorDB 최신화 완료 ==
 기존 청크 수: 0개
 최신화 후 최종 청크 수: 10222개
----------------------------------------
 == 세부 변경 내역 총 10222개 ==
 - 유지 : 0개
 - 수정 : 0개
 - 삭제 : 0개
 - 추가 : 10222개
----------------------------------------
적재 스크립트 실행 완료!


In [3]:
# 데이터 추가 삽입 로직이 추가된 DB 적재 스크립트 실행
import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 10222개의 청크 확인
적재 전 Chroma DB 청크 수: 10222개
업데이트할 내용이 없습니다. VectorDB 내 모든 데이터가 이미 최신 상태입니다.
== VectorDB 최신화 완료 ==
 기존 청크 수: 10222개
 최신화 후 최종 청크 수: 10222개
----------------------------------------
 == 세부 변경 내역 총 10222개 ==
 - 유지 : 10222개
 - 수정 : 0개
 - 삭제 : 0개
 - 추가 : 0개
----------------------------------------
적재 스크립트 실행 완료!


In [1]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [6]:
# 데이터 추가 (샘플데이터 3개 추가해보기)
# yaml 에서 chunk 경로 sample데이터로, 로컬 DB에 3개가 추가되는지 실험

import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 3개의 청크 확인
적재 전 Chroma DB 청크 수: 10222개


더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_042, 청크 100개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_070, 청크 132개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_051, 청크 63개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_076, 청크 92개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_057, 청크 112개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_098, 청크 62개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_059, 청크 91개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_100, 청크 89개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_001, 청크 61개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_078, 청크 105개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_012, 청크 145개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_077, 청크 88개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_006, 청크 70개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_003, 청크 142개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_084, 청크 119개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_013, 청크 155개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_007, 청크 63개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_023, 청크 73개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_008, 청크 278개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_087, 청크 151개)
더 이상 존재하지 않는 문서 삭제 중 (doc_id: doc_048, 청크 146개)
더 

In [7]:
# 데이터 추가 (샘플데이터 3개 추가해보기)
# yaml 에서 chunk 경로 sample데이터로, 로컬 DB에 3개가 추가되는지 실험

import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 3개의 청크 확인
적재 전 Chroma DB 청크 수: 3개
더 이상 존재하지 않는 구버전 청크 1개 삭제 중...
추가(변경)된 청크 1개 저장 중
[1/1] 0 ~ 1번째 대상 처리 중


== VectorDB 최신화 완료 ==
 기존 청크 수: 3개
 최신화 후 최종 청크 수: 3개
----------------------------------------
 == 세부 변경 내역 총 4개 ==
 - 유지 : 2개
 - 수정 : 0개
 - 삭제 : 1개
 - 추가 : 1개
----------------------------------------
적재 스크립트 실행 완료!


In [4]:
# 데이터 추가 (샘플데이터 3개 추가해보기)
# yaml 에서 chunk 경로 sample데이터로, 로컬 DB에 3개가 추가되는지 실험

import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 3개의 청크 확인
적재 전 Chroma DB 청크 수: 3개
더 이상 존재하지 않는 문서 삭제 중 (doc_id: sample_rfp123, 청크 1개)
추가(변경)된 청크 1개 저장 중
[1/1] 0 ~ 1번째 대상 처리 중
== VectorDB 최신화 완료 ==
 기존 청크 수: 3개
 최신화 후 최종 청크 수: 3개
----------------------------------------
 == 세부 변경 내역 총 4개 ==
 - 유지 : 2개
 - 수정 : 0개
 - 삭제 : 1개
 - 추가 : 1개
----------------------------------------
적재 스크립트 실행 완료!


In [8]:
# 데이터 추가 (샘플데이터 3개 추가해보기)
# yaml 에서 chunk 경로 sample데이터로, 로컬 DB에 3개가 추가되는지 실험

import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 2개의 청크 확인
적재 전 Chroma DB 청크 수: 3개
더 이상 존재하지 않는 문서 삭제 중 (doc_id: sample_rfp123, 청크 1개)
업데이트할 내용이 없습니다. VectorDB 내 모든 데이터가 이미 최신 상태입니다.
== VectorDB 최신화 완료 ==
 기존 청크 수: 3개
 최신화 후 최종 청크 수: 2개
----------------------------------------
 == 세부 변경 내역 총 3개 ==
 - 유지 : 2개
 - 수정 : 0개
 - 삭제 : 1개
 - 추가 : 0개
----------------------------------------
적재 스크립트 실행 완료!


In [9]:
# 데이터 추가 (샘플데이터 3개 추가해보기)
# yaml 에서 chunk 경로 sample데이터로, 로컬 DB에 3개가 추가되는지 실험

import sys
sys.path.append("../")
from scripts.build_api_vectordb import build_db


print("데이터 적재를 시작합니다...")
build_db()
print("적재 스크립트 실행 완료!")

데이터 적재를 시작합니다...
JSONL 청크 파일 로드 완료: 10222개의 청크 확인
적재 전 Chroma DB 청크 수: 2개
더 이상 존재하지 않는 문서 삭제 중 (doc_id: sample_rfp, 청크 2개)
추가(변경)된 청크 10222개 저장 중
[1/11] 0 ~ 1000번째 대상 처리 중
[2/11] 1000 ~ 2000번째 대상 처리 중
[3/11] 2000 ~ 3000번째 대상 처리 중
[4/11] 3000 ~ 4000번째 대상 처리 중
[5/11] 4000 ~ 5000번째 대상 처리 중
[6/11] 5000 ~ 6000번째 대상 처리 중
[7/11] 6000 ~ 7000번째 대상 처리 중
[8/11] 7000 ~ 8000번째 대상 처리 중
[9/11] 8000 ~ 9000번째 대상 처리 중
[10/11] 9000 ~ 10000번째 대상 처리 중
[11/11] 10000 ~ 10222번째 대상 처리 중
== VectorDB 최신화 완료 ==
 기존 청크 수: 2개
 최신화 후 최종 청크 수: 10222개
----------------------------------------
 == 세부 변경 내역 총 10224개 ==
 - 유지 : 0개
 - 수정 : 0개
 - 삭제 : 2개
 - 추가 : 10222개
----------------------------------------
적재 스크립트 실행 완료!


# 17. [Retrieval 1] OpenAI Chroma 재시작·영속성 검증

In [9]:
import sqlite3

# VM Chroma DB의 sqlite 파일 직접 연결
conn = sqlite3.connect("/data/processed/vector_db/openai/chroma.sqlite3")
cursor = conn.cursor()

# 임베딩 테이블 개수 조회
cursor.execute("SELECT count(*) FROM embeddings;")
count = cursor.fetchone()[0]

print(f"기존 Vector DB 파일 내 저장된 임베딩 개수: {count}개")
conn.close()

기존 Vector DB 파일 내 저장된 임베딩 개수: 10222개


In [3]:
# 새 프로세스에서 Chroma DB 생성 시도
# 터미널 환경에서 실행 하듯 셀에서 진행

!python ../scripts/build_api_vectordb.py


JSONL 청크 파일 로드 완료: 10222개의 청크 확인
적재 전 Chroma DB 청크 수: 10222개
업데이트할 내용이 없습니다. VectorDB 내 모든 데이터가 이미 최신 상태입니다.
== VectorDB 최신화 완료 ==
 기존 청크 수: 10222개
 최신화 후 최종 청크 수: 10222개
----------------------------------------
 == 세부 처리 결과 ==
 - 유지 : 10222개
 - 수정 : 0개
 - 삭제 : 0개
 - 추가 : 0개
----------------------------------------


커널 재시작 후 /data/processed/vector_db/openai(VM 경로)의 VectorDB 상태 확인 (영속성 검증)

In [ ]:
import sys
sys.path.append("../")
import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 설정 로드 및 검색
with open("../config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

top_k = config["retrieval"]["top_k"] # 현재 yaml 기준 5

retriever = OpenAIChromaRetriever(config)
query = "대학과 관련된 사업 정보를 알려줘"
results_before = retriever.search(query, top_k = top_k)

print("=== 재시작 전 검색 결과 ===")
for i, res in enumerate(results_before):
    print(f"[{i+1}등] doc_id: {res.doc_id} | score: {res.score:.4f}")
    print(f"text: {res.text[:150]}...")

=== 재시작 전 검색 결과 ===
[1등] doc_id: doc_021 | score: 0.3447
text: 신본부__________________________________ 사업과 관련하여 사업수행업체인 ________________은 본 사업을 완료함에 따라 다음의 누출 금지 정보를 이상없이 파기하였음을 확인합니다. ① 대학 소유 정보시스템의 내ㆍ외부 IP주소 현황 ② ...
[2등] doc_id: doc_008 | score: 0.3445
text:  있도록 직위별로 기재함 사업책임자 부문 부문 부문 -268- [ 서식 15 ] 참여인력 현황표 분야별 소속기관 성명 직급 근무경력 소속 최종학교 참여율(%) 담당업무 사업책임자 부문 부문 부문 부문 부문 부문 부문 부문 부문 부문 ※ 사업책임자(Project Mana...
[3등] doc_id: doc_013 | score: 0.3189
text: [사전공개용]제 안 요 청 서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한 자료로써 실제 입찰공고 시 사업금액, 과업내용, 평가항목, 제출서류 등은 변경될 수 있으니 반드시 확인하시기 바랍니다. 2023. 06. 담당성명소 속전화번호e-ma...
[4등] doc_id: doc_093 | score: 0.3163
text:  보관 금지 ○ 사업 완료 후 업체 소유 PC·서버의 하드디스크·휴대용 저장매체 등 전자기록 저장매체는 국가정보원장이 안전성을 검증한 삭제 S/W로 완전 삭제 후 반출 ○ 용역사업 관련자료 회수 및 삭제조치 후 업체에게 복사본 등 용역사업관련 자료를 보유하고 있지 않다...
[5등] doc_id: doc_045 | score: 0.3143
text: 대상정보’에 대한 보안관리 계획을 사업제안서에 기재하여야 하며, 해당 정보 누출시 대학은 국가계약법 시행령 제76조에 따라 사업자를 부정당업체로 등록한다. < 누출금지 대상정보 > ① 서영대학교 소유 정보시스템의 내ㆍ외부 IP주소 현황 ② 세부 정보시스템 구성현황 및

In [ ]:
# 3. 새로운 프로세스에서 DB 로드 및 검색 (영속성 검증)
import sys
sys.path.append("../")
import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 설정 로드 및 검색
with open("../config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

top_k = config["retrieval"]["top_k"] # 현재 yaml 기준 5

retriever_new = OpenAIChromaRetriever(config)
query = "대학과 관련된 사업 정보를 알려줘"
results_after = retriever_new.search(query, top_k = top_k)


print("=== 재시작 후 검색 결과 ===")
for i, res in enumerate(results_after):
    print(f"[{i+1}등] doc_id: {res.doc_id} | score: {res.score:.4f}")
    print(f"text: {res.text[:150]}...")

=== 재시작 후 검색 결과 ===
[1등] doc_id: doc_021 | score: 0.3447
text: 신본부__________________________________ 사업과 관련하여 사업수행업체인 ________________은 본 사업을 완료함에 따라 다음의 누출 금지 정보를 이상없이 파기하였음을 확인합니다. ① 대학 소유 정보시스템의 내ㆍ외부 IP주소 현황 ② ...
[2등] doc_id: doc_008 | score: 0.3445
text:  있도록 직위별로 기재함 사업책임자 부문 부문 부문 -268- [ 서식 15 ] 참여인력 현황표 분야별 소속기관 성명 직급 근무경력 소속 최종학교 참여율(%) 담당업무 사업책임자 부문 부문 부문 부문 부문 부문 부문 부문 부문 부문 ※ 사업책임자(Project Mana...
[3등] doc_id: doc_013 | score: 0.3189
text: [사전공개용]제 안 요 청 서 본 제안요청서는 입찰참여의 균등한 기회 제공을 위해 규격을 공개하기 위한 자료로써 실제 입찰공고 시 사업금액, 과업내용, 평가항목, 제출서류 등은 변경될 수 있으니 반드시 확인하시기 바랍니다. 2023. 06. 담당성명소 속전화번호e-ma...
[4등] doc_id: doc_093 | score: 0.3163
text:  보관 금지 ○ 사업 완료 후 업체 소유 PC·서버의 하드디스크·휴대용 저장매체 등 전자기록 저장매체는 국가정보원장이 안전성을 검증한 삭제 S/W로 완전 삭제 후 반출 ○ 용역사업 관련자료 회수 및 삭제조치 후 업체에게 복사본 등 용역사업관련 자료를 보유하고 있지 않다...
[5등] doc_id: doc_045 | score: 0.3143
text: 대상정보’에 대한 보안관리 계획을 사업제안서에 기재하여야 하며, 해당 정보 누출시 대학은 국가계약법 시행령 제76조에 따라 사업자를 부정당업체로 등록한다. < 누출금지 대상정보 > ① 서영대학교 소유 정보시스템의 내ㆍ외부 IP주소 현황 ② 세부 정보시스템 구성현황 및

커널 재시작 후, 같은 질문 같은 top_k로 실험했을때 같은 결과를 가져옴

# 34. [Retrieval 1] OpenAI MMR 검색 실험

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [12]:
import sys
sys.path.append("../")

import time
import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever

# 1. 설정 로드
with open("../config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 공통 테스트 질문 세트
test_queries = [
    # 사업의 예산과 무상유지보수기간 명시
    "차세대 포털·학사 정보시스템 구축 사업의 총 예산액과 무상유지보수기간은 어떻게 규정되어 있나요?",

    # 고려대학교의 사업 정보 및 기대 효과
    "고려대학교 차세대 포털·학사 정보시스템 구축 사업의 추진 배경과 주요 구축 범위, 그리고 기대 효과를 종합적으로 설명해 주세요",

    # 구체적 항목에 대한 근거
    "서울시립대학교의 '대입전형 자료', '학적 정보', '학생 활동 정보' 데이터는 각각 어느 부서에서 담당하여 수집 및 관리하고 있나요?",

    # 입찰 방식 및 계약 방법 근거
    "경상북도 봉화군의 '재난통합관리시스템 고도화 사업'의 계약 방법(입찰 방식)과 관련 법령 근거는 무엇인가요?",

    # 사업 범위 / 핵심 구축 근거
    "한국사학진흥재단의 '대학재정정보시스템(기본재산 및 기채 사후관리) 고도화' 사업의 주요 기능 개선 범위는 어떻게 되나요?"
]

def run_experiment(method: str):
    print(f"\n === [실험 모드: {method.upper()}] ===")
    config["retrieval"]["search_method"] = method
    retriever = OpenAIChromaRetriever(config)
    
    total_time = 0
    for idx, query in enumerate(test_queries, 1):
        start_time = time.time()
        results = retriever.search(query, top_k=5)
        elapsed = time.time() - start_time
        total_time += elapsed
        
        print(f"\nQ{idx}: {query}")
        print(f"지연시간: {elapsed:.3f}초")
        for i, res in enumerate(results):
            
            # MMR은 score가 0.0으로 반환됨
            score_str = f"{res.score:.4f}" if res.score > 0 else "N/A(MMR)"
            
            metadata = getattr(res, 'metadata', {})
            doc_title = metadata.get('title', '제목없음')
            
            print(f"  [{i+1}등] {res.doc_id} / Title: {doc_title} / Chunk: {res.chunk_id} / Score: {score_str}")
            print(f"  - 텍스트: {res.text[:150]}...")  # 텍스트가 너무 길어 출력이 지저분해지지 않게 앞부분만 표시
            
    print(f"\n✅ {method.upper()} 모드 평균 지연시간: {total_time / len(test_queries):.3f}초")

# 실험 실행
run_experiment("similarity")
run_experiment("mmr")


 === [실험 모드: SIMILARITY] ===



Q1: 차세대 포털·학사 정보시스템 구축 사업의 총 예산액과 무상유지보수기간은 어떻게 규정되어 있나요?
지연시간: 0.676초
  [1등] doc_008 / Title: 차세대 포털·학사 정보시스템 구축사업 / Chunk: doc_008_chunk_0006 / Score: 0.3541
  - 텍스트: 12개월 라. 사업예산 : 11,270,000,000원 (V.A.T 포함, 3년 분할 지급) - 2024학년도 약 30% 지급 - 2025학년도 약 40% 지급 - 2026학년도 약 30% 지급 마. 입찰 및 계약 방법: 제한 경쟁 입찰(협상에 의한 계약) 바. 본 사...
  [2등] doc_080 / Title: 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고) / Chunk: doc_080_chunk_0001 / Score: 0.3239
  - 텍스트: 차세대 ERP시스템 구축 사업 제 안 요 청 서 2024. 6. 목 차 Ⅰ. 사업 개요 1. 사업명 謸 2. 사업기간 蕜 3. 사업예산 蕜 4. 계약방법 蕜 Ⅱ. 주요 시스템 구축·운영 현황 1. ERP 시스템 絈 2. 그룹웨어 시스템 淰 Ⅲ. 사업 추진방안 1. 차세...
  [3등] doc_008 / Title: 차세대 포털·학사 정보시스템 구축사업 / Chunk: doc_008_chunk_0005 / Score: 0.3182
  - 텍스트: 6 Ⅵ. 제안서 작성 방법 ································································································· 242 1. 제안서 효력 ·······················...
  [4등] doc_013 / Title: [사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역 / Chunk: doc_013_chunk_0005 / Score: 0.2628
  - 텍스트:  ❍ 사업기간 : 계약체결일로부터 5개월 ❍ 사 업 비 : 금242,900,000

In [11]:
# mmr 순수 답변
import sys
sys.path.append("../")

import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever
from openai import OpenAI
from dotenv import load_dotenv
import os

# 1. 환경변수 및 설정 파일 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. 설정 로드
with open("../config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. Retriever 초기화 (YAML 설정 자동 반영: similarity 또는 mmr)
retriever = OpenAIChromaRetriever(config)
current_method = config["retrieval"]["search_method"]
print(f"현재 적용된 검색 방식: [{current_method.upper()}]")

# 3. 테스트할 공통 질문 리스트 중 Q2로 실험
test_questions = [
    # "Q1. 차세대 포털 구축 사업의 예산 규모와 무상 유지보수 기간은 어떻게 되나요?",
    "Q2. 고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.",
    # "Q3. 서울시립대 사업 내에서 입학처, 교무처, 학생처 등 데이터 담당 부서는 각각 어디인가요?",
    # "Q4. 봉화군 재난통합관제센터 구축 관련 계약 방법 및 적용 법령은 무엇인가요?",
    # "Q5. 한국사학진흥재단 시스템의 기능 개선 범위에 대해 설명해주세요."
]

# 4. 검색 및 순수 답변(Raw Generation) 실행
generation_config = config.get("generation", {})
model_name = generation_config.get("model", "gpt-5-nano") 

for i, query in enumerate(test_questions, 1):
    print(f"\n{'='*50}")
    print(f"[{i}] 질문: {query}")
    print(f"{'='*50}")
    
    # Retriever로 문서 검색 수행
    retrieved_results = retriever.search(query=query)
    
    # 검색된 청크 컨텍스트 구성
    context = ""
    print(f"검색된 문서 개수: {len(retrieved_results)}개")
    for r in retrieved_results:

        metadata = getattr(r, 'metadata', {})
        doc_title = metadata.get('title', '제목없음')
        
        print(f" - [{r.doc_id}] {doc_title} (score: {r.score:.4f})")
        context += f"\n\n[Doc ID: {r.doc_id} / Title: {doc_title}]\n{r.text}"
    
    # 순수 답변
    raw_prompt = f"""다음 문서를 참고하여 질문에 답변하시오.

[참고 문서]
{context}

[질문]
{query}
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": raw_prompt}]
    )
    
    raw_answer = response.choices[0].message.content
    print(f"\n[순수 모델 답변]:\n{raw_answer}\n")

현재 적용된 검색 방식: [SIMILARITY]

[1] 질문: Q2. 고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.
검색된 문서 개수: 5개
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.4694)
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.4030)
 - [doc_017] 예약발매시스템 개량 ISMP 용역 (score: 0.3582)
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.3438)
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.3206)

[순수 모델 답변]:
다음은 고려대학교 차세대 포털·학사 정보시스템 구축 사업의 주요 범위와 특징을 종합한 요약입니다.

1) 주요 범위 (사업 범위의 핵심 내용)
- 대상 시스템 구성
  - 포털시스템: 포털을 통한 통합 로그인(SSO), 통합/지능형 검색, 마이페이지 기능 등
  - 학사 시스템: 학사 업무 연계 및 데이터 연계, 포털과 학사 시스템의 상호 연결 강화
  - 모바일 앱: 포털·학사 시스템 기능을 모바일에서도 이용 가능하도록 시스템 현황에 맞춘 모바일 앱 기능 제공
- 시스템 통합 및 연계
  - 포털과 각 업무시스템 간의 역할 재정립 및 서비스 메뉴 정리
  - 타 부서 자체 도입 시스템 포함 여부를 고려한 전체 서비스 구조의 재설계 및 연계 표준화
  - 포털이 주요 관문으로 작동하도록 다층적 연결고리 강화
- 개인화 및 접근성 향상
  - 신분에 따른 개인화된 포털 서비스 제공(개인화된 마이페이지 및 맞춤형 기능)
  - 포털의 접근성 개선 및 통합 채널 강화
- 데이터/정보화 체계
  - 정보시스템 간 데이터 연계 및 표준화 체계 마련
  - 데이터 거버넌스 및 정보화 비전/전략에 맞춘 개선 요구사항 도출과 반영
- 기술·아키텍처 및 신기술 도입
  - 인프라 아키텍처 구성 및 구축
  - 빅데이터, AI 등 4차 산업 

In [8]:
# mmr 순수 답변
import sys
sys.path.append("../")

import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever
from openai import OpenAI
from dotenv import load_dotenv
import os

# 1. 환경변수 및 설정 파일 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. 설정 로드
with open("../config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. Retriever 초기화 (YAML 설정 자동 반영: similarity 또는 mmr)
retriever = OpenAIChromaRetriever(config)
current_method = config["retrieval"]["search_method"]
print(f"현재 적용된 검색 방식: [{current_method.upper()}]")

# 3. 테스트할 공통 질문 리스트 중 Q2로 실험
test_questions = [
    # "Q1. 차세대 포털 구축 사업의 예산 규모와 무상 유지보수 기간은 어떻게 되나요?",
    "Q2. 고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.",
    # "Q3. 서울시립대 사업 내에서 입학처, 교무처, 학생처 등 데이터 담당 부서는 각각 어디인가요?",
    # "Q4. 봉화군 재난통합관제센터 구축 관련 계약 방법 및 적용 법령은 무엇인가요?",
    # "Q5. 한국사학진흥재단 시스템의 기능 개선 범위에 대해 설명해주세요."
]

# 4. 검색 및 순수 답변(Raw Generation) 실행
generation_config = config.get("generation", {})
model_name = generation_config.get("model", "gpt-5-nano") 

for i, query in enumerate(test_questions, 1):
    print(f"\n{'='*50}")
    print(f"[{i}] 질문: {query}")
    print(f"{'='*50}")
    
    # Retriever로 문서 검색 수행
    retrieved_results = retriever.search(query=query)
    
    # 검색된 청크 컨텍스트 구성
    context = ""
    print(f"검색된 문서 개수: {len(retrieved_results)}개")
    for r in retrieved_results:

        metadata = getattr(r, 'metadata', {})
        doc_title = metadata.get('title', '제목없음')
        
        print(f" - [{r.doc_id}] {doc_title} (score: {r.score:.4f})")
        context += f"\n\n[Doc ID: {r.doc_id} / Title: {doc_title}]\n{r.text}"
    
    # 순수 답변
    raw_prompt = f"""다음 문서를 참고하여 질문에 답변하시오.

[참고 문서]
{context}

[질문]
{query}
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": raw_prompt}]
    )
    
    raw_answer = response.choices[0].message.content
    print(f"\n[순수 모델 답변]:\n{raw_answer}\n")

현재 적용된 검색 방식: [MMR]

[1] 질문: Q2. 고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.
검색된 문서 개수: 5개
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.0000)
 - [doc_017] 예약발매시스템 개량 ISMP 용역 (score: 0.0000)
 - [doc_024] 강릉어선안전조업국 상황관제시스템 구축 (score: 0.0000)
 - [doc_059] 수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입찰 공고 (score: 0.0000)
 - [doc_048] [재공고]차세대 통합정보시스템(ERP) 구축 (score: 0.0000)

[순수 모델 답변]:
다음은 고려대학교 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합한 설명입니다.

1) 주요 범위 (사업 범위의 핵심 영역)
- 차세대 포털 시스템의 구축 및 학사정보시스템과의 통합
  - 전사 포털과 학사정보서비스를 하나의 통합 플랫폼으로 설계·구축합니다.
- 포털 구성의 다중화 및 포틀릿 기반 UI 구성
  - 역할과 목적에 따라 다수의 포털을 생성하고, 포틀릿(패널/카드 형태의 기능 단위) 템플릿을 통해 화면을 구성하는 체계를 포함합니다.
- 초기화면 및 관리 기능의 구체화
  - 전사 포털의 초기화면과 업무 포털의 초기화면 기능 정의 및 구현, 그리고 포털 관리 기능의 마련(권한/역할 관리, 포털 템플릿/레이아웃 관리 등)을 포함합니다.
- 공지/배너/빠른 접근 채널의 제공
  - 포털 내 공지사항, 전파사항, 배너 등을 공유 채널로 활용하는 기능과 관리 체계를 구축합니다.
- 업무 포털과의 연계 및 업무 프로세스 지원
  - 전자결재, 메일, PAMS 등 주요 업무 포털과의 연계를 지원하여 업무 효율성을 높입니다.
- 포털 간의 연계 및 표준화된 아키텍처 구축
  - 포털 간 상호연계가 용이한 아키텍처를 마련하고, 시스템 간 인터페이스의 표준화를 도모합니다.
- 학사정보시스템과의 원

## MMR lambda_mult = 0.7로 답변 재생성

In [12]:
# mmr (0.7) 순수 답변 => 관련성 확률 up
import sys
sys.path.append("../")

import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever
from openai import OpenAI
from dotenv import load_dotenv
import os

# 1. 환경변수 및 설정 파일 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. 설정 로드
with open("../config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. Retriever 초기화 (YAML 설정 자동 반영: similarity 또는 mmr)
retriever = OpenAIChromaRetriever(config)
current_method = config["retrieval"]["search_method"]
print(f"현재 적용된 검색 방식: [{current_method.upper()}]")

# 3. 테스트할 공통 질문 리스트 중 Q2로 실험
test_questions = [
    # "Q1. 차세대 포털 구축 사업의 예산 규모와 무상 유지보수 기간은 어떻게 되나요?",
    # Q2. 
    "고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.",
    # "Q3. 서울시립대 사업 내에서 입학처, 교무처, 학생처 등 데이터 담당 부서는 각각 어디인가요?",
    # "Q4. 봉화군 재난통합관제센터 구축 관련 계약 방법 및 적용 법령은 무엇인가요?",
    # "Q5. 한국사학진흥재단 시스템의 기능 개선 범위에 대해 설명해주세요."
]

# 4. 검색 및 순수 답변(Raw Generation) 실행
generation_config = config.get("generation", {})
model_name = generation_config.get("model", "gpt-5-nano") 

for i, query in enumerate(test_questions, 1):
    print(f"\n{'='*50}")
    print(f"[{i}] 질문: {query}")
    print(f"{'='*50}")
    
    # Retriever로 문서 검색 수행
    retrieved_results = retriever.search(query=query)
    
    # 검색된 청크 컨텍스트 구성
    context = ""
    print(f"검색된 문서 개수: {len(retrieved_results)}개")
    for r in retrieved_results:

        metadata = getattr(r, 'metadata', {})
        doc_title = metadata.get('title', '제목없음')
        
        print(f" - [{r.doc_id}] {doc_title} (score: {r.score:.4f})")
        context += f"\n\n[Doc ID: {r.doc_id} / Title: {doc_title}]\n{r.text}"
    
    # 순수 답변
    raw_prompt = f"""다음 문서를 참고하여 질문에 답변하시오.

[참고 문서]
{context}

[질문]
{query}
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": raw_prompt}]
    )
    
    raw_answer = response.choices[0].message.content
    print(f"\n[순수 모델 답변]:\n{raw_answer}\n")

현재 적용된 검색 방식: [MMR]

[1] 질문: 고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.
검색된 문서 개수: 5개
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.0000)
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.0000)
 - [doc_080] 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고) (score: 0.0000)
 - [doc_055] 한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도화 용역 (score: 0.0000)
 - [doc_061] 2024년도 평택시 버스정보시스템(BIS) 구축사업 (score: 0.0000)

[순수 모델 답변]:
다음은 고려대학교 차세대 포털·학사 정보시스템 구축 사업의 주요 범위와 특징을 요약한 내용입니다.

핵심 요약
- 목적: 교내 정보시스템의 관문이자 사용자 맞춤 서비스를 제공하는 차세대 포털을 재설계·구축하고, 포털을 중심으로 학사 정보시스템과의 연계성과 이용 편의성을 제고하는 것.
- 주요 방향: 단일 로그인 체계 도입, 다양한 디바이스를 아우르는 유연한 구조, 정보의 일관성 유지, 사용자 그룹별 맞춤 포털 제공, 행정시스템과의 연계 강화, 보안·서비스 관리 기능의 체계화.

주요 범위 (Scope)
- 포털 재설계 및 구축
  - 포털은 교내 정보시스템으로의 관문 역할과 사용자 맞춤 서비스를 제공하는 대전제 하에 설계.
  - 단일 로그인(SSO)으로 교내 모든 정보시스템에 진입 및 서비스 제공.
  - 구조의 유연성 확보: 다양한 디바이스 호환, 다국어 확대, 개인형 포털 구성 가능.
  - 정보의 일관성 유지: 포털 웹 서비스와 모바일 서비스에서 제공하는 정보의 일관성 보장.
- 포털 공통 기능 및 사용자 그룹 관리
  - 사용자별 맞춤형 포털 시스템 구축: 학생, 교원, 직원, 연구원 등으로 행정시스템과 연계해 포털 사용자 그룹 관리.
  - 계정/권한 정책을 반영한 접근

## lambda_mult = 0.9 로 답변 재생성

In [3]:
# mmr (0.8) 순수 답변 => 관련성 확률 up
import sys
sys.path.append("../")

import yaml
from src.openai_chroma_retriever import OpenAIChromaRetriever
from openai import OpenAI
from dotenv import load_dotenv
import os

# 1. 환경변수 및 설정 파일 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. 설정 로드
with open("../config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 2. Retriever 초기화 (YAML 설정 자동 반영: similarity 또는 mmr)
retriever = OpenAIChromaRetriever(config)
current_method = config["retrieval"]["search_method"]
print(f"현재 적용된 검색 방식: [{current_method.upper()}]")

# 3. 테스트할 공통 질문 리스트 중 Q2로 실험
test_questions = [
    # "Q1. 차세대 포털 구축 사업의 예산 규모와 무상 유지보수 기간은 어떻게 되나요?",
    # Q2. 
    "고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.",
    # "Q3. 서울시립대 사업 내에서 입학처, 교무처, 학생처 등 데이터 담당 부서는 각각 어디인가요?",
    # "Q4. 봉화군 재난통합관제센터 구축 관련 계약 방법 및 적용 법령은 무엇인가요?",
    # "Q5. 한국사학진흥재단 시스템의 기능 개선 범위에 대해 설명해주세요."
]

# 4. 검색 및 순수 답변(Raw Generation) 실행
generation_config = config.get("generation", {})
model_name = generation_config.get("model", "gpt-5-nano") 

for i, query in enumerate(test_questions, 1):
    print(f"\n{'='*50}")
    print(f"[{i}] 질문: {query}")
    print(f"{'='*50}")
    
    # Retriever로 문서 검색 수행
    retrieved_results = retriever.search(query=query)
    
    # 검색된 청크 컨텍스트 구성
    context = ""
    print(f"검색된 문서 개수: {len(retrieved_results)}개")
    for r in retrieved_results:

        metadata = getattr(r, 'metadata', {})
        doc_title = metadata.get('title', '제목없음')
        
        print(f" - [{r.doc_id}] {doc_title} (score: {r.score:.4f})")
        context += f"\n\n[Doc ID: {r.doc_id} / Title: {doc_title}]\n{r.text}"
    
    # 순수 답변
    raw_prompt = f"""다음 문서를 참고하여 질문에 답변하시오.

[참고 문서]
{context}

[질문]
{query}
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": raw_prompt}]
    )
    
    raw_answer = response.choices[0].message.content
    print(f"\n[순수 모델 답변]:\n{raw_answer}\n")

현재 적용된 검색 방식: [MMR]

[1] 질문: 고려대 차세대 포털 시스템 구축 사업의 주요 범위와 특징을 종합적으로 설명해주세요.
검색된 문서 개수: 5개
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.0000)
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.0000)
 - [doc_008] 차세대 포털·학사 정보시스템 구축사업 (score: 0.0000)
 - [doc_017] 예약발매시스템 개량 ISMP 용역 (score: 0.0000)
 - [doc_080] 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고) (score: 0.0000)

[순수 모델 답변]:
다음은 고려대학교 차세대 포털·학사 정보시스템 구축 사업의 주요 범위와 특징을 요약한 내용입니다.

1) 사업 개요 요약
- 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업
- 사업기간: 계약일로부터 24개월
- 무상유지보수기간: 사업 종료일로부터 12개월
- 예산: 11,270,000,000원(부가가치세 포함, 3년 분할 지급)
  - 2024학년도 약 30% 지급
  - 2025학년도 약 40% 지급
  - 2026학년도 약 30% 지급
- 주요 서비스 구성 대상: 포털, 학사 시스템, 모바일 앱 시스템의 기능 및 연계 서비스
- 인프라: 하드웨어 및 시스템 소프트웨어 현황에 대한 검토 및 반영

2) 사업 범위의 핵심 내용
- 차세대 포털 및 학사 정보시스템 구축 전 과정
  - 포털과 학사 시스템의 기능을 통합적으로 설계·구축
  - 포털이 행정시스템의 관문 역할을 하도록 재설계
- 서비스 메뉴 정비 및 이중 관리 해소
  - 다년간 누적 운영으로 남아 있는 구 메뉴/신 메뉴의 중복 및 타 부서 도입 시스템과의 기능 중복 제거
  - 포털과 각 업무시스템 간의 역할 재정립 및 메뉴 정리
- 개인화 및 사용자 중심 서비스 강화
  - 교내 구성원 각자의 신분에 맞는 개인화 포털 제공
  - 맞춤형 서비스 및